# FlowMatch – Internal Capacity & Growth Marketplace
**Hybrid Matching and Recommendation Engine**

This notebook implements the core matching engine for FlowMatch. Given a pool of employees with spare capacity and a set of short-term projects, it surfaces the **top 3 recommended matches** for any project — complete with an LLM-generated **Fit Rationale** and **Stretch Rationale** for each pairing.

**Algorithm design**
- *Fit score* — cosine similarity between an employee's current skills and the project's requirements (embedded via `text-embedding-3-small`)
- *Stretch score* — cosine similarity between an employee's target/growth skills and the skills the project develops
- *Clone penalty* — soft penalty when fit is suspiciously perfect (avoids matching someone with a task identical to their day job)
- *Composite score* — 40 % fit + 60 % stretch − clone penalty (biased toward developmental matches per design brief)

**How to run**
1. Run *Section 1* once per session (installs packages, sets API key)
2. Run *Section 2* to load the data
3. Run *Section 3* once to embed all employees (cached in memory)
4. Use *Section 4* to match any single project, or *Section 5* for a full batch run
5. Use *Section 6* to manually evaluate a sample of matches


---
## Section 1 – Setup

In [ ]:
# Install dependencies (run once per Colab session)
!pip install openai pandas numpy openpyxl --quiet

In [ ]:
import os

REPO_URL = "https://github.com/znantwan/flow-match.git"
REPO_DIR = "/content/flow-match"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    # Pull latest changes if already cloned
    !git -C {REPO_DIR} pull --quiet

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("Repository ready.")

In [ ]:
from openai import OpenAI
from getpass import getpass

# Paste your OpenAI API key when prompted (it will not be displayed or saved)
OPENAI_API_KEY = getpass("Enter your OpenAI API key: ")
client = OpenAI(api_key=OPENAI_API_KEY)
print("OpenAI client initialised.")

---
## Section 2 – Load Data

Data is read from the `data/` folder in the repo. Both **Excel** (`.xlsx`) and **CSV** (`.csv`) files are supported — the loader detects the format automatically.

To use your own data, replace the files in the repo and re-run the clone cell in Section 1.

In [ ]:
import pandas as pd
import glob, os

DATA_DIR = os.path.join(REPO_DIR, "data")

def load_table(data_dir, keywords):
    """
    Find and load a spreadsheet whose filename contains any of the given keywords.
    Searches .xlsx first, then .xls, then .csv. Case-insensitive.
    """
    all_files = os.listdir(data_dir)
    for ext, reader in [(".xlsx", pd.read_excel), (".xls", pd.read_excel), (".csv", pd.read_csv)]:
        for fname in all_files:
            if fname.lower().endswith(ext) and any(k in fname.lower() for k in keywords):
                full = os.path.join(data_dir, fname)
                print(f"  Loading {fname}")
                return reader(full)
    raise FileNotFoundError(
        f"No file found in {data_dir} matching keywords {keywords} with extension .xlsx/.csv"
    )

employees_df = load_table(DATA_DIR, ["employee"])
projects_df  = load_table(DATA_DIR, ["project"])

employees = employees_df.fillna("").to_dict(orient="records")
projects  = projects_df.fillna("").to_dict(orient="records")

print(f"\nLoaded {len(employees)} employees and {len(projects)} projects.")
print(f"Employee columns: {list(employees_df.columns)}")
print(f"Project columns:  {list(projects_df.columns)}")

In [ ]:
# Validate that required columns exist
REQUIRED_EMPLOYEE_COLS = [
    "employee_id", "name", "department", "role",
    "current_skills", "target_skills",
    "available_hours_per_week", "availability_end_date",
]
REQUIRED_PROJECT_COLS = [
    "project_id", "project_name", "manager_name", "department",
    "description", "required_skills",
    "hours_needed", "duration_weeks", "industry_context",
]

def check_columns(df, required, label):
    missing = [c for c in required if c not in df.columns]
    if missing:
        print(f"WARNING — {label} is missing columns: {missing}")
        print(f"  Found: {list(df.columns)}")
    else:
        print(f"  {label}: all required columns present.")

check_columns(employees_df, REQUIRED_EMPLOYEE_COLS, "employees")
check_columns(projects_df,  REQUIRED_PROJECT_COLS,  "projects")

In [ ]:
# Preview employee data
employees_df[[
    "employee_id", "name", "department", "role",
    "current_skills", "target_skills", "available_hours_per_week"
]].head(10)

In [ ]:
# Preview project data
projects_df[[
    "project_id", "project_name", "manager_name", "department",
    "required_skills", "hours_needed", "duration_weeks"
]].head(10)

---
## Section 3 – Embed All Employees

We embed each employee's **current skills** and **target skills** separately and cache the vectors in memory. This only calls the OpenAI API once per session; matching any number of projects afterwards is fast.

In [ ]:
from flowmatch import build_employee_embeddings

print(f"Embedding {len(employees)} employees (2 API calls each) — this takes ~30 seconds...")
current_embeds, target_embeds = build_employee_embeddings(client, employees)
print("Done. All employee embeddings cached.")

---
## Section 4 – Match a Single Project

Change `PROJECT_ID` to any project ID from the data, then run the cell.

In [ ]:
import json
from flowmatch import rank_employees_for_project

PROJECT_ID = "PROJ001"  # ← change this to any project_id

project = next(p for p in projects if p["project_id"] == PROJECT_ID)

print(f"\nMatching: {project['project_name']}")
print(f"Manager:  {project['manager_name']} ({project['department']})")
print(f"Required: {project['required_skills']}")
print(f"Duration: {project['hours_needed']} hrs over {project['duration_weeks']} weeks")
print("\nFinding top 3 matches...\n")

matches = rank_employees_for_project(
    client, project, employees, current_embeds, target_embeds, top_n=3
)

In [ ]:
def display_matches(project, matches):
    print("=" * 70)
    print(f"PROJECT: {project['project_name']}  [{project['project_id']}]")
    print(f"         {project['description'][:100]}...")
    print("=" * 70)
    for i, m in enumerate(matches, 1):
        s = m["scores"]
        warning = "  ⚠ Capacity mismatch" if s.get("capacity_warning") else ""
        print(f"\nMatch #{i}: {m['name']}  ({m['role']}, {m['department']}){warning}")
        print(f"  Available: {m['available_hours_per_week']} hrs/week until {m['availability_end_date']}")
        print(f"  Scores  →  Fit: {s['fit_score']:.1%}  |  Stretch: {s['stretch_score']:.1%}  |  Composite: {s['composite_score']:.1%}")
        print(f"  FIT RATIONALE:")
        print(f"    {m['fit_rationale']}")
        print(f"  STRETCH RATIONALE:")
        print(f"    {m['stretch_rationale']}")
    print("\n" + "=" * 70)

display_matches(project, matches)

In [ ]:
# Full structured JSON output for integration or logging
output = {
    "project_id": project["project_id"],
    "project_name": project["project_name"],
    "matches": matches,
}
print(json.dumps(output, indent=2))

---
## Section 5 – Batch Run (All Projects)

Runs the matching engine over all 15 projects and collects results.  
Note: this makes 2 OpenAI embedding calls + 3 GPT-4o-mini calls **per project**, so ~15 × 5 = 75 API calls total. Takes about 3–5 minutes.

In [ ]:
all_results = []

for proj in projects:
    print(f"Matching {proj['project_id']}: {proj['project_name']} ...")
    proj_matches = rank_employees_for_project(
        client, proj, employees, current_embeds, target_embeds, top_n=3
    )
    all_results.append({
        "project_id": proj["project_id"],
        "project_name": proj["project_name"],
        "matches": proj_matches,
    })

print(f"\nBatch complete. {len(all_results)} projects processed.")

In [ ]:
# Summary table: top-1 match for every project
summary_rows = []
for r in all_results:
    top = r["matches"][0]
    summary_rows.append({
        "project_id": r["project_id"],
        "project_name": r["project_name"],
        "top_match": top["name"],
        "role": top["role"],
        "fit_score": f"{top['scores']['fit_score']:.1%}",
        "stretch_score": f"{top['scores']['stretch_score']:.1%}",
        "composite_score": f"{top['scores']['composite_score']:.1%}",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
# Save full results to JSON
OUTPUT_PATH = "/content/flowmatch_results.json"
with open(OUTPUT_PATH, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Results saved to {OUTPUT_PATH}")

# Download from Colab
from google.colab import files
files.download(OUTPUT_PATH)

---
## Section 6 – Manual Evaluation

The evaluation plan calls for manually assessing 10 sample matches.  
This section iterates through a random sample and prompts you to score each pairing.

In [ ]:
import random
import ipywidgets as widgets
from IPython.display import display, clear_output

EVAL_SAMPLE_SIZE = 10  # number of project-match pairs to evaluate

# Build flat list of (project, match) pairs from top-1 of each project
eval_pairs = []
for r in all_results:
    proj = next(p for p in projects if p["project_id"] == r["project_id"])
    for m in r["matches"][:1]:  # top-1 only; change to [:3] for all matches
        eval_pairs.append((proj, m))

sample = random.sample(eval_pairs, min(EVAL_SAMPLE_SIZE, len(eval_pairs)))
eval_scores = []

print(f"Evaluating {len(sample)} project-match pairs. Score each on a 1-5 scale.")

In [ ]:
def evaluate_pair(idx, proj, match):
    clear_output(wait=True)
    s = match["scores"]
    print(f"--- Evaluation {idx + 1}/{len(sample)} ---")
    print(f"PROJECT:  {proj['project_name']}")
    print(f"EMPLOYEE: {match['name']}  ({match['role']}, {match['department']})")
    print(f"Scores    Fit: {s['fit_score']:.1%}  |  Stretch: {s['stretch_score']:.1%}  |  Composite: {s['composite_score']:.1%}")
    print(f"\nFIT: {match['fit_rationale']}")
    print(f"\nSTRETCH: {match['stretch_rationale']}")
    print("\n" + "-" * 60)

    fit_q = input("Fit quality (1=poor, 5=excellent): ").strip()
    stretch_q = input("Stretch quality (1=poor, 5=excellent): ").strip()
    clone_q = input("Is this a clone match? (y/n — same as employee's day job): ").strip()
    notes = input("Notes (optional): ").strip()

    eval_scores.append({
        "project_id": proj["project_id"],
        "project_name": proj["project_name"],
        "employee_id": match["employee_id"],
        "employee_name": match["name"],
        "fit_score_algo": s["fit_score"],
        "stretch_score_algo": s["stretch_score"],
        "composite_score_algo": s["composite_score"],
        "fit_quality_human": fit_q,
        "stretch_quality_human": stretch_q,
        "is_clone": clone_q.lower() == "y",
        "notes": notes,
    })

for i, (proj, match) in enumerate(sample):
    evaluate_pair(i, proj, match)

clear_output(wait=True)
print("Evaluation complete!")
eval_df = pd.DataFrame(eval_scores)
display(eval_df)

In [ ]:
# Summary statistics
eval_df["fit_quality_human"] = pd.to_numeric(eval_df["fit_quality_human"], errors="coerce")
eval_df["stretch_quality_human"] = pd.to_numeric(eval_df["stretch_quality_human"], errors="coerce")

print("=== Evaluation Summary ===")
print(f"Pairs evaluated:           {len(eval_df)}")
print(f"Avg human fit quality:     {eval_df['fit_quality_human'].mean():.2f} / 5")
print(f"Avg human stretch quality: {eval_df['stretch_quality_human'].mean():.2f} / 5")
print(f"Clone matches identified:  {eval_df['is_clone'].sum()}")
print(f"Avg algo composite score:  {eval_df['composite_score_algo'].mean():.1%}")

# Correlation between algorithm score and human judgment
corr_fit    = eval_df["fit_score_algo"].corr(eval_df["fit_quality_human"])
corr_stretch = eval_df["stretch_score_algo"].corr(eval_df["stretch_quality_human"])
print(f"\nCorrelation (algo fit vs human fit):        {corr_fit:.2f}")
print(f"Correlation (algo stretch vs human stretch): {corr_stretch:.2f}")

In [ ]:
# Save evaluation results
EVAL_PATH = "/content/flowmatch_evaluation.csv"
eval_df.to_csv(EVAL_PATH, index=False)
print(f"Evaluation saved to {EVAL_PATH}")
files.download(EVAL_PATH)

---
## Section 7 – Score Distribution Analysis

Quick diagnostic charts to understand how the algorithm is distributing scores across all project-match pairs.

In [ ]:
import matplotlib.pyplot as plt

# Flatten all matches for analysis
flat_rows = []
for r in all_results:
    for m in r["matches"]:
        flat_rows.append({
            "project_id": r["project_id"],
            "employee_id": m["employee_id"],
            "fit_score": m["scores"]["fit_score"],
            "stretch_score": m["scores"]["stretch_score"],
            "clone_penalty": m["scores"]["clone_penalty"],
            "composite_score": m["scores"]["composite_score"],
        })

flat_df = pd.DataFrame(flat_rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("FlowMatch Score Distributions (Top-3 Matches Across All Projects)")

axes[0].hist(flat_df["fit_score"], bins=15, color="steelblue", edgecolor="white")
axes[0].set_title("Fit Score")
axes[0].set_xlabel("Score")

axes[1].hist(flat_df["stretch_score"], bins=15, color="darkorange", edgecolor="white")
axes[1].set_title("Stretch Score")
axes[1].set_xlabel("Score")

axes[2].hist(flat_df["composite_score"], bins=15, color="mediumseagreen", edgecolor="white")
axes[2].set_title("Composite Score")
axes[2].set_xlabel("Score")

plt.tight_layout()
plt.savefig("/content/score_distributions.png", dpi=150)
plt.show()

print(flat_df.describe().round(3))

In [ ]:
# Fit vs Stretch scatter — ideal matches are in the upper-right (high both)
# Clone candidates appear as high-fit / low-stretch dots near the right edge
fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(
    flat_df["fit_score"],
    flat_df["stretch_score"],
    c=flat_df["composite_score"],
    cmap="RdYlGn",
    s=80,
    alpha=0.8,
    edgecolors="grey",
    linewidths=0.5,
)
plt.colorbar(scatter, label="Composite Score")
ax.set_xlabel("Fit Score (baseline competence)")
ax.set_ylabel("Stretch Score (growth alignment)")
ax.set_title("Fit vs Stretch for All Top-3 Match Pairs")
ax.axvline(0.85, color="red", linestyle="--", alpha=0.5, label="Clone penalty threshold")
ax.legend()
plt.tight_layout()
plt.savefig("/content/fit_vs_stretch.png", dpi=150)
plt.show()